# Identifying Quasi-Periodic Eruption (QPE) Candidates

### STATiX Variable Source Catalogue — Pouliasis et al. (A&A)

This notebook reproduces the QPE candidate search described in Sect. "Quasi-periodic eruptions — QPEs" of the paper, using **only the published catalogue** (`STATIX_VC_published.fits`).



## Important notes on catalogue columns

A few quantities used in the original internal analysis are **not stored directly** in the published table and are reconstructed here from what *is* available:

| Needed quantity | Internal column | Published-table equivalent |
|---|---|---|
| Galactic latitude `b` | `b` (precomputed) | Computed on the fly from `RA_X`/`DEC_X` via `astropy.coordinates` |
| "Has a good pn detection" | `fl_PN_CORR` | Approximated as "row has a valid, non-empty `pn_Z_SCORE`/`pn_LC_BB`" |
| Source classification | `final_class` | Unified from `CLASS`, falling back to `CLASS_RF` where `CLASS == -1` |
| Bayesian-block Z-scores | `PN_Z_SCORE_SIG` | `pn_Z_SCORE` (same quantity, renamed) |
| Bayesian-block start/end/counts | `PN_LC_BB` | `pn_LC_BB` (same quantity, renamed) |

Light-curve visualisation also differs from the internal version: the original code read per-frame timestamps from archival data cubes (`Cube(...)`). The published table does not carry raw cube files, so all plots here are built purely from the **Bayesian-block start/end times** stored in `pn_LC_BB`, which is sufficient to reproduce the block-level (step-function) light curve shown in the paper's figures.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from astropy.table import Table, unique
from astropy.time import Time
from astropy.coordinates import SkyCoord
import astropy.units as u
from scipy.ndimage import label

## 1. Load the published catalogue

In [ ]:
out = Table.read("STATIX_VC_published.fits")
print(f"Loaded {len(out)} detections with {len(out.colnames)} columns.")

## 2. Compute Galactic latitude

The published table stores equatorial coordinates (`RA_X`, `DEC_X`) but not Galactic latitude directly. We compute it here with `astropy.coordinates`.

In [ ]:
coords = SkyCoord(ra=np.asarray(out["RA_X"]) * u.deg,
                   dec=np.asarray(out["DEC_X"]) * u.deg,
                   frame="icrs")
out["b_deg"] = coords.galactic.b.deg

print("Galactic latitude computed. Example values:")
print(out["b_deg"][:5])

## 3. Build a unified source classification (`final_class`)

`CLASS` is the primary classification; where it is `-1` (unclassified), we fall back to the Random-Forest classification `CLASS_RF`. Both use the same coding convention:

| Value | Meaning |
|---|---|
| -1 | unclassified (by both methods) |
| 0 | secure Galactic |
| 1 | likely Galactic |
| 2 | likely extragalactic |
| 3 | secure extragalactic |

In [ ]:
out["final_class"] = np.where(out["CLASS"] == -1, out["CLASS_RF"], out["CLASS"])

n_still_unclassified = np.sum(out["final_class"] == -1)
print(f"Sources still unclassified after CLASS_RF fallback: {n_still_unclassified}")

## 4. Select the pn / high-Galactic-latitude parent sample

Following the paper, the QPE search is restricted to:
1. Detections with a valid pn light curve (approximated here as: non-empty `pn_Z_SCORE` and `pn_LC_BB` arrays).
2. Galactic latitude $|b| \geq 10\deg$, to reduce contamination from stellar flares.

In [ ]:
def has_valid_pn(row):
    """True if this row has a usable pn Z-score / Bayesian-block array."""
    z_arr = np.ma.filled(np.atleast_1d(row["pn_Z_SCORE"]), np.nan).astype(float)
    bb_arr = np.ma.filled(np.atleast_2d(row["pn_LC_BB"]), np.nan)
    has_z = np.any(~np.isnan(z_arr))
    has_bb = np.any(~np.isnan(bb_arr).all(axis=1))
    return bool(has_z and has_bb)


pn_mask = np.array([has_valid_pn(row) for row in out])
pn = out[pn_mask]
print(f"Detections with a usable pn light curve: {len(pn)}")

b10 = pn[np.abs(pn["b_deg"]) >= 10]
print(f"...and |b| >= 10 deg: {len(b10)}")

## 5. Justify the Z-score threshold

The paper adopts a Z-score threshold of 8.2, corresponding to the $3\sigma$ quantile of the median per-source Z-score distribution. We reproduce that diagnostic here for reference; small numerical differences from the paper's exact threshold are expected since this is recomputed independently from the released columns.

In [ ]:
median_z = []
for row in pn["pn_Z_SCORE"]:
    arr = np.ma.filled(np.atleast_1d(row), np.nan).astype(float)
    median_z.append(np.nanmedian(arr) if np.any(~np.isnan(arr)) else np.nan)
median_z = np.array(median_z)

plt.figure(figsize=(6, 4))
plt.hist(median_z[~np.isnan(median_z)], bins=150, color="lightgray", edgecolor="black")
plt.yscale("log")
plt.xlabel("Median pn Z-score per source")
plt.ylabel("Number of sources")
plt.title("Distribution of median pn Z-score (cf. Z-score threshold = 8.2)")
plt.tight_layout()
plt.show()

print("Adopted QPE search threshold: Z-score > 8.2 (paper value, 3-sigma quantile).")

## 6. Eruption-finding algorithm

For each detection, we scan its Bayesian-block Z-score sequence for contiguous runs of blocks exceeding the threshold. Following the paper, we require at least 3 consecutive blocks above threshold to suppress spurious, short-lived fluctuations, and record the peak block of each run as the candidate eruption.

In [ ]:
Z_THRESHOLD = 8.2
MIN_CONSECUTIVE_BLOCKS = 3

found_events = []

for row in b10:
    z_raw = row["pn_Z_SCORE"]
    z_scores = np.ma.filled(np.atleast_1d(z_raw), 0).astype(float).flatten()

    t_bounds = np.ma.filled(np.atleast_2d(row["pn_LC_BB"]), np.nan)
    valid_bb = ~np.isnan(t_bounds).any(axis=1)
    t_bounds = t_bounds[valid_bb]
    z_scores = z_scores[: len(t_bounds)]

    if len(t_bounds) != len(z_scores) or len(z_scores) == 0:
        continue

    t_mid = (t_bounds[:, 0] + t_bounds[:, 1]) / 2.0

    qpe_mask = z_scores > Z_THRESHOLD
    labeled_array, num_features = label(qpe_mask, structure=np.ones(3))

    for i in range(1, num_features + 1):
        idx = np.where(labeled_array == i)[0]
        if len(idx) < MIN_CONSECUTIVE_BLOCKS:
            continue

        peak_idx = idx[np.argmax(z_scores[idx])]

        found_events.append({
            "group_id": int(row["group_id"]),
            "PEAK_TIME": t_mid[peak_idx],
            "MAX_Z_SCORE": z_scores[peak_idx],
            "DETID": row["DETID"],
            "TOTAL_ISLANDS": num_features,
            "final_class": row["final_class"],
            "b_deg": row["b_deg"],
            "N_BINS": len(idx),
        })

final_table = Table(found_events) if found_events else Table()
print(f"Candidate QPE events found: {len(final_table)}")


## 7. Classify candidate events

We split the candidate events by their unified `final_class`, then reduce to unique sources (`group_id`) within each population.

In [ ]:
q5 = final_table

extra = q5[(q5["final_class"] == 2) | (q5["final_class"] == 3)]
stars = q5[(q5["final_class"] == 0) | (q5["final_class"] == 1)]
unclas = q5[q5["final_class"] == -1]

print(f"Candidate events: extragalactic={len(extra)}, Galactic={len(stars)}, unclassified={len(unclas)}")

starsU = unique(stars, keys="group_id")
extraU = unique(extra, keys="group_id")
#unclasU = unique(unclas, keys="group_id")

print()
print(f"Unique extragalactic sources: {len(extraU)}   (paper: 14 excluding sources with Random Forest classification)")
print(f"Unique Galactic sources     : {len(starsU)}   (paper: 25 excluding sources with Random Forest classification)")
#print(f"Unique unclassified sources : {len(unclasU)}   (paper: 4)")

## 8. Visualise extragalactic QPE candidates

The function below plots the pn Bayesian-block light curve of every observation belonging to a given `group_id`, placed on a common, continuous time axis. As noted above, this uses **only** the block start/end times in `pn_LC_BB` (no external cube files), since that is what the published table provides.

Times are stored in XMM Mission Elapsed Time (MET) seconds, referenced to 1998-01-01 00:00:00 (TT).

In [ ]:
MET_REF = Time("1998-01-01 00:00:00", scale="tt")


def _clean_bb(bb_row):
    """Return a clean (N, 6) array of valid pn Bayesian blocks from a pn_LC_BB row."""
    arr = np.ma.filled(np.atleast_2d(bb_row), np.nan)
    valid = ~np.isnan(arr).any(axis=1)
    arr = arr[valid]
    return arr if arr.shape[0] > 0 else None


def _reconstruct_frame_times(lc_bb):
    """
    Approximate per-frame times from Bayesian-block start/end times,
    assuming frames are equally spaced within each block. Needed because
    the published pn_LC array stores only counts, with no timestamps.
    """
    frame_times = []
    for jj in range(lc_bb.shape[0]):
        start, end, n_frames_block = lc_bb[jj, 0], lc_bb[jj, 1], lc_bb[jj, 2]
        n_frames_block = int(n_frames_block) if n_frames_block > 0 else 1
        dt = (end - start) / n_frames_block
        block_times = start + dt * (np.arange(n_frames_block) + 0.5)
        frame_times.append(block_times)
    return np.concatenate(frame_times)


def plot_group_lc_published(out, group_id, n_rows=4, n_cols=5, label=None):
    """
    Plot the pn raw light curve + Bayesian-block representation for every
    observation belonging to `group_id`, placed on a common, continuous
    time axis. Uses only OBSID, pn_LC, and pn_LC_BB from the published
    `out` table -- no external cube files.

    Raw per-frame timestamps are reconstructed from the Bayesian-block
    start/end times (frames assumed equally spaced within a block), since
    the published table does not store per-frame times directly. This is
    an approximation.
    """
    group_rows = out[out["group_id"] == group_id]
    if len(group_rows) == 0:
        print(f"No rows found for group_id={group_id}")
        return

    all_data = []
    for row in group_rows:
        bb = _clean_bb(row["pn_LC_BB"])
        if bb is None:
            continue

        raw_lc = np.ma.filled(np.atleast_2d(row["pn_LC"]), np.nan)
        mask_lc = ~np.isnan(raw_lc[:, 0])
        lc = raw_lc[mask_lc]

        bb_sorted = bb[np.argsort(bb[:, 0])]
        frame_times = _reconstruct_frame_times(bb_sorted)

        n_use = min(len(frame_times), len(lc))
        frame_times = frame_times[:n_use]
        lc = lc[:n_use]

        t_start_seconds = np.min(bb_sorted[:, 0])
        obs_date_str = (MET_REF + t_start_seconds / 86400.0).datetime.strftime("%Y-%m-%d")

        all_data.append({
            "obsid": row["OBSID"],
            "obs_date_str": obs_date_str,
            "bb": bb_sorted,
            "lc": lc,
            "frame_times": frame_times,
            "abs_time_val": t_start_seconds,
        })

    if not all_data:
        print(f"No valid Bayesian-block data found for group_id={group_id}")
        return

    all_data = sorted(all_data, key=lambda x: x["abs_time_val"])
    t0 = all_data[0]["abs_time_val"]

    rates = [d["bb"][:, 3] / np.where(d["bb"][:, 2] == 0, 1, d["bb"][:, 2]) for d in all_data]
    ymax_bb = max(r.max() for r in rates)
    ymax_raw = max((d["lc"][:, 0].max() if d["lc"].size > 0 else 0) for d in all_data)
    ymax = max(ymax_bb, ymax_raw) * 1.1

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 2.5, n_rows * 2.2))
    axes = np.atleast_1d(axes).flatten()

    for idx, info in enumerate(all_data):
        if idx >= n_rows * n_cols:
            break
        ax = axes[idx]
        bb = info["bb"]

        # --- Raw light curve (reconstructed times) ---
        if info["lc"].size > 0:
            t_days_raw = (info["frame_times"] - t0) / 86400.0
            ax.step(t_days_raw, info["lc"][:, 0], where="mid", color="red", alpha=0.4, lw=1)

        # --- Bayesian block steps ---
        for j in range(bb.shape[0]):
            s_rel = (bb[j, 0] - t0) / 86400.0
            e_rel = (bb[j, 1] - t0) / 86400.0
            rate = bb[j, 3] / (bb[j, 2] if bb[j, 2] != 0 else 1)
            ax.hlines(rate, s_rel, e_rel, colors="black", lw=1.5)

        if idx == 0 and label is not None:
            ax.text(0.05, 0.92, label, transform=ax.transAxes,
                    fontsize=10, fontweight="bold", color="blue", ha="left", va="top",
                    bbox=dict(facecolor="white", alpha=0.6, edgecolor="none"))

        ax.set_title(f"{info['obs_date_str']}\n{info['obsid']}", fontsize=7)
        ax.set_ylim(0, ymax)
        ax.xaxis.set_major_locator(ticker.MaxNLocator(nbins=3))
        ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.1f"))
        ax.tick_params(axis="both", labelsize=6)
        ax.grid(True, alpha=0.1)

    for i in range(len(all_data), n_rows * n_cols):
        axes[i].axis("off")

    fig.suptitle(f"group_id = {group_id}", fontsize=10)
    fig.supxlabel(r"Time $-$ $t_0$ [days]", fontsize=10)
    fig.supylabel("Counts / frame", fontsize=10)
    plt.tight_layout()
    plt.savefig(f"QPE_LC_{group_id}.png", dpi=150)
    plt.show()

In [ ]:
for i in range(len(starsU)):
    gid = starsU["group_id"][i]
    print(f"group_id = {gid}  (DETID = {starsU['DETID'][i]}, "
          f"islands = {starsU['TOTAL_ISLANDS'][i]}, N_BINS = {starsU['N_BINS'][i]})")
    plot_group_lc_published(out, group_id=gid, n_rows=4, n_cols=5)

## 9. Cross-checking against known, literature-confirmed QPE sources

The paper reports recovering six robust QPE sources after visual inspection, all independently confirmed in the literature: four eROSITA-discovered QPEs covered by XMM (ERO-QPE1–4), plus GSN 069 and RX J1301.9+2747. A fifth eROSITA QPE (ERO-QPE5) did not pass the pipeline's initial quality cuts.

Since this notebook works from `RA_X`/`DEC_X` positions rather than internal source names, we cross-match the extragalactic candidate list above against the published coordinates of these sources to identify which `group_id`s they correspond to.

In [ ]:
# Approximate literature coordinates (ICRS, degrees) for known QPE sources,
# derived from arcsecond-precision 2MASX/2MASS identifiers where available.
known_qpes = {
    "GSN069":   (19.78625, -34.19181),
    "RXJ1301":  (195.50058, 27.78274),
    "ERO-QPE1": (37.94646, -10.33644),
    "ERO-QPE2": (38.70300, -44.32569),
    "ERO-QPE3": (210.22213, -28.76700),
    "ERO-QPE4": (71.39083, -10.20131),
}

match_radius_arcsec = 30


def build_coord_map(candidate_table, out):
    """
    Explicit group_id -> (RA, Dec) map, aligned by construction regardless
    of unique()'s internal row ordering.
    """
    lookup_rows = unique(out[np.isin(out["group_id"], candidate_table["group_id"])], keys="group_id")
    return {
        gid: (ra, dec)
        for gid, ra, dec in zip(lookup_rows["group_id"], lookup_rows["RA_X"], lookup_rows["DEC_X"])
    }


def crossmatch_known_qpes(candidate_table, out, population_label):
    coord_map = build_coord_map(candidate_table, out)
    gids = np.asarray(candidate_table["group_id"])
    ra_lookup = np.array([coord_map[g][0] for g in gids])
    dec_lookup = np.array([coord_map[g][1] for g in gids])
    cand_coords = SkyCoord(ra=ra_lookup * u.deg, dec=dec_lookup * u.deg)

    print(f"Matches within the {population_label} QPE candidate sample "
          f"({len(candidate_table)} sources):")
    for name, (ra, dec) in known_qpes.items():
        ref = SkyCoord(ra=ra * u.deg, dec=dec * u.deg)
        sep = ref.separation(cand_coords).arcsec
        best = np.argmin(sep)
        if sep[best] <= match_radius_arcsec:
            print(f"  {name:10s} -> group_id={gids[best]}  (separation = {sep[best]:.1f} arcsec)")
        else:
            print(f"  {name:10s} -> no match within {match_radius_arcsec} arcsec "
                  f"(closest: {sep[best]:.1f} arcsec)")
    print()


crossmatch_known_qpes(extraU, out, "extragalactic")
crossmatch_known_qpes(starsU, out, "Galactic")

**Note:** the coordinates above are approximate reference values; if a source does not match within the search radius, double-check its published position directly (e.g. via SIMBAD/NED) rather than assuming a non-detection.

## 10. Combined summary figure for confirmed/candidate QPE sources

Once you have identified the `group_id`s of interest (e.g. from the cross-match above), this function lays out all of their observations together in a single figure, with custom labels, in the same style as the multi-source QPE figure in the paper.

In [ ]:
def plot_all_groups_combined_published(out, group_list, custom_labels=None, n_rows=5, n_cols=5):
    """
    Combined multi-source QPE light-curve figure, built entirely from the
    published pn_LC / pn_LC_BB representation (no external cube files).
    `group_list` is a list of group_id values; `custom_labels` is an
    optional list of the same length giving a display name per source.

    Raw per-frame timestamps are reconstructed from the Bayesian-block
    start/end times (frames assumed equally spaced within a block), since
    the published table does not store per-frame times directly.
    """
    max_panels = n_rows * n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 2.5, n_rows * 3.5))
    axes = axes.flatten()
    panel_counter = 0

    for i, group_id in enumerate(group_list):
        group_rows = out[out["group_id"] == group_id]
        if len(group_rows) == 0:
            continue

        group_data = []
        for row in group_rows:
            bb = _clean_bb(row["pn_LC_BB"])
            if bb is None:
                continue

            raw_lc = np.ma.filled(np.atleast_2d(row["pn_LC"]), np.nan)
            mask_lc = ~np.isnan(raw_lc[:, 0])
            lc = raw_lc[mask_lc]

            bb_sorted = bb[np.argsort(bb[:, 0])]
            frame_times = _reconstruct_frame_times(bb_sorted)

            n_use = min(len(frame_times), len(lc))
            frame_times = frame_times[:n_use]
            lc = lc[:n_use]

            t_start_seconds = np.min(bb_sorted[:, 0])
            obs_date_str = (MET_REF + t_start_seconds / 86400.0).datetime.strftime("%Y-%m-%d")

            group_data.append({
                "obsid": row["OBSID"],
                "obs_date_str": obs_date_str,
                "bb": bb_sorted,
                "lc": lc,
                "frame_times": frame_times,
                "abs_time_val": t_start_seconds,
            })

        if not group_data:
            continue

        group_data = sorted(group_data, key=lambda x: x["abs_time_val"])
        t0 = group_data[0]["abs_time_val"]

        ymax_bb = max((d["bb"][:, 3] / np.where(d["bb"][:, 2] == 0, 1, d["bb"][:, 2])).max()
                      for d in group_data)
        ymax_raw = max((d["lc"][:, 0].max() if d["lc"].size > 0 else 0) for d in group_data)
        ymax_group = max(ymax_bb, ymax_raw)

        for j, info in enumerate(group_data):
            if panel_counter >= max_panels:
                break
            ax = axes[panel_counter]
            bb = info["bb"]

            # --- Raw light curve (reconstructed times) ---
            if info["lc"].size > 0:
                t_days_raw = (info["frame_times"] - t0) / 86400.0
                ax.step(t_days_raw, info["lc"][:, 0], where="mid", color="red", alpha=0.4, lw=1)

            # --- Bayesian block steps ---
            for k in range(bb.shape[0]):
                s_rel = (bb[k, 0] - t0) / 86400.0
                e_rel = (bb[k, 1] - t0) / 86400.0
                rate = bb[k, 3] / (bb[k, 2] if bb[k, 2] != 0 else 1)
                ax.hlines(rate, s_rel, e_rel, colors="black", lw=1.5)

            ax.set_ylim(0, ymax_group * 1.1)
            ax.set_title(f"{info['obs_date_str']}\n{info['obsid']}", fontsize=9)

            if j == 0:
                label_text = custom_labels[i] if custom_labels and i < len(custom_labels) else f"Gr{i+1}"
                ax.text(0.05, 0.92, label_text, transform=ax.transAxes,
                        fontsize=11, fontweight="bold", color="blue", ha="left", va="top",
                        bbox=dict(facecolor="white", alpha=0.6, edgecolor="none"))

            ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=4))
            ax.xaxis.set_major_locator(ticker.MaxNLocator(nbins=3))
            ax.xaxis.set_major_formatter(ticker.FormatStrFormatter("%.1f"))
            ax.tick_params(axis="both", labelsize=9)
            ax.grid(True, alpha=0.1)

            panel_counter += 1

    for i in range(panel_counter, n_rows * n_cols):
        axes[i].axis("off")

    fig.supxlabel(r"Time $-$ $t_0$ [days]", fontsize=12, y=0.06)
    fig.supylabel("Counts / frame", fontsize=12, x=0.06)
    plt.tight_layout(rect=[0.04, 0.0, 0.95, 0.95])
    plt.savefig("QPE_all_groups_combined.pdf", dpi=400, bbox_inches="tight")
    plt.show()

## Fill in with the group_id identified above, and the corresponding display names.

In [ ]:
group_list = [2952]         
custom_labels = ["groupID 2952"]       # e.g. ["RXJ1301", "GSN069", ...]

if group_list:
    plot_all_groups_combined_published(out, group_list, custom_labels=custom_labels, n_rows=1, n_cols=5)
else:
    print("Populate group_list / custom_labels above (see Section 9) before running this cell.")